## LangChain tools
Tools in LangChain are external functions or services (e.g., search engine, calculator, API) that the agent can call while solving tasks to extend its capabilities beyond just text generation.

### Install libraries

In [ ]:
# (setup cell already installs what this notebook needs)

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Optional keys. A few notebooks call a third-party API : Tavily in demos05a,
# OpenWeatherMap in demos05b, LangSmith in demos09. load_dotenv() covers the
# local path ; in Colab there is no .env, so they are read from Secrets here.
# Missing is fine, the cell that needs one says so.
if IN_COLAB:
    try:
        from google.colab import userdata
        for _name in ("TAVILY_API_KEY", "OWM_API_KEY", "LANGSMITH_API_KEY"):
            try:
                _v = userdata.get(_name)
                if _v:
                    os.environ[_name] = _v
            except Exception:
                pass
    except ImportError:
        pass


### Simple tool (calculator)

In [1]:
from langchain.tools import tool

@tool
def multiply_numbers(x: int, y: int) -> int:
    """Multiplies two integers"""
    print("CALLED")
    return x * y

print(multiply_numbers.invoke({"x": 6, "y": 7}))

CALLED
42


C:\languages\Python311\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### Combine tools with LLM

In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def multiply_numbers(x: int, y: int) -> int:
    """Multiplies two integers"""
    print("CALLED")
    return x * y

load_dotenv()

llm = make_llm()
tools = [multiply_numbers]

tools = [multiply_numbers]

agent = create_agent(
    model=llm,         
    tools=tools
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is twice the number of people on Earth?"}]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What is twice the number of people on Earth?
================================== Ai Message ==================================
Tool Calls:
  multiply_numbers (call_Yg8Fu3IISkW64IUoapoE8L5F)
 Call ID: call_Yg8Fu3IISkW64IUoapoE8L5F
  Args:
    x: 8
    y: 2
CALLED
================================= Tool Message =================================
Name: multiply_numbers

16
================================== Ai Message ==================================

Twice the number of people on Earth is approximately 16 billion.


### Tool - simple knowledge database

In [8]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

llm = make_llm()

@tool
def company_info(name: str) -> str:
    """Returns basic company information based on name."""
    companies = {
        "OpenAI": "Artificial intelligence company, creator of ChatGPT.",
        "LangChain": "A library that makes it easy to build applications with language models.",
        "Tesla": "Manufacturer of electric cars and batteries."
    }
    return companies.get(name, "I don't know this company.")

tools = [company_info]

agent = create_agent(
    model=llm,         
    tools=tools
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me what Tesla does? You should use the tools available and nothing else"}]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me what Tesla does? You should use the tools available and nothing else
================================== Ai Message ==================================
Tool Calls:
  company_info (call_ifUgKRYycqcSoVB4EZlWljV2)
 Call ID: call_ifUgKRYycqcSoVB4EZlWljV2
  Args:
    name: Tesla
================================= Tool Message =================================
Name: company_info

Manufacturer of electric cars and batteries.
================================== Ai Message ==================================

Tesla is a manufacturer of electric cars and batteries.


### Available tools in LangChain
List of available tools: https://python.langchain.com/docs/integrations/tools/

In [ ]:
!pip install wikipedia tavily-python langchain-tavily

### Direct use of tools

In [15]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# Initialize search tools
# Tavily API key: https://app.tavily.com/home
search = TavilySearch(max_results=3)

# Direct use of tools
query = "The latest news about the UK Prime Minister."
results = search.invoke({"query": query})

# TavilySearch returns a dict; the hits live under "results"
hits = results["results"] if isinstance(results, dict) else results

print(f"Search results for query: '{query}'")
for i, result in enumerate(hits, 1):
    print(f"\n{i}. {result.get('title', 'Missing title')}")
    print(f"   URL: {result.get('url', 'Missing URL')}")
    print(f"   Content: {result.get('content', 'Missing content')[:200]}...")

Search results for query: 'The latest news about the UK Prime Minister.'

1. UK Prime Minister Keir Starmer under pressure | DW News - YouTube
   URL: https://www.youtube.com/watch?v=AZqFu75-QgU
   Content: U.K. Prime Minister Keir Starmer is facing a career-defining decision: step down or fight a possible challenge from Labour Party rival Andy...

2. Politics latest: Burnham hints at tax changes - as he's ... - Sky News
   URL: https://news.sky.com/story/politics-latest-burnham-starmer-labour-tories-badenoch-farage-12593360
   Content: Andy Burnham has said he is yet to choose his chancellor before becoming, as is widely expected, the next prime minister. He also wants a "complete rethink"...

3. The Torture Chamber of British Politics Crushes Its Latest Prime ...
   URL: https://www.newyorker.com/news/the-lede/the-torture-chamber-of-british-politics-crushes-its-latest-prime-minister
   Content: Sam Knight writes about Keir Starmer, who has now become the sixth Prime Minister over the

### Agent with Tavily Search Tool

In [13]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent

llm = make_llm()

# List of tools with search engine
tools = [search]

# Agent with internet access
agent_with_search = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful research assistant. Use web search to find accurate, up-to-date information.",
)

# Example query requiring internet search
response = agent_with_search.invoke({
    "messages": [{"role": "user", "content": "Who is the current Prime Minister of UK and how old is he/she?"}]
})
print(f"\nAgent answer:\n{response['messages'][-1].content}")


Agent answer:
The current Prime Minister of the United Kingdom is **Sir Keir Starmer**, who has been in office since **July 5, 2024**. He was born on **September 2, 1962**, which makes him **61 years old** as of now, and he will turn **62** in September 2024.


### Agent with Youtube Search Tool

In [ ]:
!pip install youtube-search 

In [5]:
from langchain_community.tools import YouTubeSearchTool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

youtube = YouTubeSearchTool()

llm = make_llm()
agent = create_agent(
    model=llm,
    tools=[youtube],
    system_prompt="You help users find relevant YouTube videos using the youtube_search tool.",
)

result = agent.invoke(
    {"messages": [HumanMessage(content="Find some videos about LangGraph tutorials")]}
)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Find some videos about LangGraph tutorials
================================== Ai Message ==================================
Tool Calls:
  youtube_search (call_gc4o6GHrxNkNKAB7EhCGT3c8)
 Call ID: call_gc4o6GHrxNkNKAB7EhCGT3c8
  Args:
    query: LangGraph tutorials
================================= Tool Message =================================
Name: youtube_search

['https://www.youtube.com/watch?v=cUfLrn3TM3M&pp=ygUTTGFuZ0dyYXBoIHR1dG9yaWFscw%3D%3D', 'https://www.youtube.com/watch?v=qAF1NjEVHhY&t=197s&pp=ygUTTGFuZ0dyYXBoIHR1dG9yaWFscw%3D%3D']
================================== Ai Message ==================================

Here are some videos about LangGraph tutorials:

1. [LangGraph Tutorial Video 1](https://www.youtube.com/watch?v=cUfLrn3TM3M&pp=ygUTTGFuZ0dyYXBoIHR1dG9yaWFscw%3D%3D)
2. [LangGraph Tutorial Video 2](https://www.youtube.com/watch?v=qAF1NjEVHhY&t=197s&pp=ygUTTGFuZ0dyYXBoIHR1dG9yaWFscw%3D%3